In [ ]:
from google.colab import files
data = files.upload()
print(data)

Here we import the libraries to process the data before training and testing 

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder  # Encodes categorical data (e.g., strings) into numerical format.
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
# mean_squared_error < Measures the average squared difference between the actual and predicted values.
# r2_score Measures < how well the regression model fits the data. Values range from 0 (poor fit) to 1 (perfect fit).
from sklearn.impute import SimpleImputer

# Load datasets
df = pd.read_csv('heart_disease_uci.csv')

# Preview the expanded dataset
df.head()


,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


filling the null columns as well as  excluding outliers

In [4]:
import pandas as pd

df = df.copy()

# ---------------------------
num_cols = ['trestbps', 'chol', 'thalch', 'oldpeak', 'ca']
cat_cols = ['sex', 'dataset', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']


for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)


for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)


for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]

# ---------------------------
df.reset_index(drop=True, inplace=True)



/tmp/ipykernel_16107/266800181.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
/tmp/ipykernel_16107/266800181.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try 

In [5]:
df.isnull().sum()

,0
id,0
age,0
sex,0
dataset,0
cp,0
trestbps,0
chol,0
fbs,0
restecg,0
thalch,0


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 589 entries, 0 to 588
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        589 non-null    int64  
 1   age       589 non-null    int64  
 2   sex       589 non-null    object 
 3   dataset   589 non-null    object 
 4   cp        589 non-null    object 
 5   trestbps  589 non-null    float64
 6   chol      589 non-null    float64
 7   fbs       589 non-null    bool   
 8   restecg   589 non-null    object 
 9   thalch    589 non-null    float64
 10  exang     589 non-null    bool   
 11  oldpeak   589 non-null    float64
 12  slope     589 non-null    object 
 13  ca        589 non-null    float64
 14  thal      589 non-null    object 
 15  num       589 non-null    int64  
dtypes: bool(2), float64(5), int64(3), object(6)
memory usage: 65.7+ KB


transforming the data to the way a model understands (0 or 1)

In [7]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

cols = ['sex','cp','fbs','restecg','exang','slope','thal','dataset']

for col in cols:
    df[col] = le.fit_transform(df[col])

In [8]:
df.head()

,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,1,0,3,145.0,233.0,1,0,150.0,0,2.3,0,0.0,0,0
1,4,37,1,0,2,130.0,250.0,0,1,187.0,0,3.5,0,0.0,1,0
2,5,41,0,0,1,130.0,204.0,0,0,172.0,0,1.4,2,0.0,1,0
3,6,56,1,0,1,120.0,236.0,0,1,178.0,0,0.8,2,0.0,1,0
4,8,57,0,0,0,120.0,354.0,0,1,163.0,1,0.6,2,0.0,1,0


In [9]:
x = df[["age","sex","cp","trestbps","chol","fbs","restecg","thalch","exang","oldpeak","slope","ca","thal"]]
y = df["num"]

dividing the data into training and testing sets to prepare for predicting

In [10]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# Building, training, and evaluating a Random Forest Classifier.
# This script initializes the model with specific hyperparameters, fits it to the training data, 
# and computes comprehensive performance metrics including Accuracy, F1-Score, and Confusion Matrix.

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix
)
model = RandomForestClassifier(n_estimators=250,max_depth=10,random_state=42)

model.fit(x_train, y_train)


# Predictions
y_pred = model.predict(x_test)
y_proba = model.predict_proba(x_test)[:, 1]  # probability for ROC-AUC

# Metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_pred, average='weighted'))
print("F1-score:", f1_score(y_test, y_pred, average='weighted'))

print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.6864406779661016
Precision: 0.6679624318703872
Recall: 0.6864406779661016
F1-score: 0.6458363500767537

Classification Report:
               precision    recall  f1-score   support

           0       0.72      0.93      0.81        67
           1       0.59      0.49      0.53        35
           2       0.50      0.12      0.20         8
           3       1.00      0.17      0.29         6
           4       0.00      0.00      0.00         2

    accuracy                           0.69       118
   macro avg       0.56      0.34      0.37       118
weighted avg       0.67      0.69      0.65       118


Confusion Matrix:
 [[62  5  0  0  0]
 [18 17  0  0  0]
 [ 2  5  1  0  0]
 [ 3  1  1  1  0]
 [ 1  1  0  0  0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

In [ ]:
# XGBoost Classifier Pipeline: Implementation and Evaluation.
# Configured with Gradient Boosting hyperparameters (n_estimators=250, learning_rate=0.1).
# Evaluates model performance using standard classification metrics and a Confusion Matrix.

In [12]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix
)

# Model
xgb_model = XGBClassifier(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

# Train
xgb_model.fit(x_train, y_train)

# Predictions
y_pred = xgb_model.predict(x_test)
y_proba = xgb_model.predict_proba(x_test)[:, 1]

# Metrics
print("=== XGBoost ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_pred, average='weighted'))
print("F1-score:", f1_score(y_test, y_pred, average='weighted'))
#print("ROC-AUC:", roc_auc_score(y_test, y_proba, multi_class='ovr'))

print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [10:54:04] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


=== XGBoost ===
Accuracy: 0.6779661016949152
Precision: 0.6368193952939716
Recall: 0.6779661016949152
F1-score: 0.6449122794104568

Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.93      0.82        67
           1       0.58      0.43      0.49        35
           2       0.50      0.25      0.33         8
           3       0.25      0.17      0.20         6
           4       0.00      0.00      0.00         2

    accuracy                           0.68       118
   macro avg       0.41      0.35      0.37       118
weighted avg       0.64      0.68      0.64       118


Confusion Matrix:
 [[62  5  0  0  0]
 [18 15  0  2  0]
 [ 2  4  2  0  0]
 [ 2  2  1  1  0]
 [ 0  0  1  1  0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

In [ ]:
# Implementing a Logistic Regression Pipeline with Data Scaling.
# The pipeline ensures that the features are standardized using StandardScaler 
# before being passed to the Logistic Regression model for training and evaluation.

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Pipeline (Scaling + Model)
log_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

# Train
log_model.fit(x_train, y_train)

# Predictions
y_pred = log_model.predict(x_test)
y_proba = log_model.predict_proba(x_test)[:, 1]

# Metrics
print("=== Logistic Regression ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_pred, average='weighted'))
print("F1-score:", f1_score(y_test, y_pred, average='weighted'))
#print("ROC-AUC:", roc_auc_score(y_test, y_proba))

print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

=== Logistic Regression ===
Accuracy: 0.6864406779661016
Precision: 0.6312870332093151
Recall: 0.6864406779661016
F1-score: 0.6502775274246096

Classification Report:
               precision    recall  f1-score   support

           0       0.76      0.93      0.83        67
           1       0.57      0.49      0.52        35
           2       0.50      0.25      0.33         8
           3       0.00      0.00      0.00         6
           4       0.00      0.00      0.00         2

    accuracy                           0.69       118
   macro avg       0.36      0.33      0.34       118
weighted avg       0.63      0.69      0.65       118


Confusion Matrix:
 [[62  5  0  0  0]
 [16 17  1  1  0]
 [ 1  5  2  0  0]
 [ 3  2  1  0  0]
 [ 0  1  0  1  0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

In [ ]:
# Hyperparameter Tuning for XGBoost using RandomizedSearchCV.
# This process automates the search for the optimal combination of parameters
# like learning_rate, max_depth, and n_estimators to maximize model performance 
# while preventing overfitting through 5-fold cross-validation.

In [14]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report
import numpy as np

xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 1],
    'colsample_bytree': [0.7, 0.8, 1]
}

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=20,              # عدد التجارب
    scoring='f1_weighted',  # أو accuracy
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(x_train, y_train)

print("Best Params:", random_search.best_params_)
print("Best Score:", random_search.best_score_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [10:54:41] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best Params: {'subsample': 1, 'n_estimators': 200, 'max_depth': 10, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
Best Score: 0.6563465835161024


In [15]:
best_model = random_search.best_estimator_

y_pred = best_model.predict(x_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.71      0.93      0.81        67
           1       0.58      0.40      0.47        35
           2       0.50      0.25      0.33         8
           3       0.33      0.17      0.22         6
           4       0.00      0.00      0.00         2

    accuracy                           0.67       118
   macro avg       0.43      0.35      0.37       118
weighted avg       0.63      0.67      0.63       118



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Exhaustive Hyperparameter Tuning using GridSearchCV.
# This script performs a systematic search through all specified parameter combinations 
# to identify the optimal configuration for the XGBoost model using 5-fold cross-validation.

In [16]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [5, 6, 7],
    'learning_rate': [0.05, 0.1]
}

grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring='f1_weighted',
    cv=5,
    verbose=1,
    n_jobs=-1
)

grid_search.fit(x_train, y_train)

print("Best Params:", grid_search.best_params_)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [10:55:01] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best Params: {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 200}


# Integrating a Pipeline with GridSearchCV for optimized Logistic Regression.
# This approach automates feature scaling and hyperparameter tuning simultaneously, 
# preventing data leakage and finding the best combination of 'C' and 'solver'.

In [17]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000))
])

param_grid = {
    'clf__C': [0.01, 0.1, 1, 10],
    'clf__solver': ['lbfgs', 'liblinear']
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)
grid.fit(x_train, y_train)

print("Best Params:", grid.best_params_)

Best Params: {'clf__C': 1, 'clf__solver': 'liblinear'}


In [18]:
!pip install flask pyngrok

# Flask Web Application for Heart Disease Prediction.
# Features:
# - Single Prediction: User inputs features via an HTML form with real-time validation.
# - Batch Prediction: Upload CSV files to get predictions for multiple records at once.
# - Risk Scoring: Categorizes risk levels based on model probability outputs.
# - Deployment: Configured to run on Google Colab using port forwarding.

In [19]:
import pandas as pd
import numpy as np
from flask import Flask, request, render_template_string
import traceback

app = Flask(__name__)

# ========== OPTIONS ==========
OPTIONS = {
    'sex': [('1', 'ذكر'), ('0', 'أنثى')],
    'cp': [('0', 'Typical Angina'), ('1', 'Atypical'), ('2', 'Non-anginal'), ('3', 'Asymptomatic')],
    'fbs': [('1', 'نعم'), ('0', 'لا')],
    'restecg': [('0', 'Normal'), ('1', 'Abnormal'), ('2', 'LVH')],
    'exang': [('1', 'نعم'), ('0', 'لا')],
    'slope': [('0', 'Up'), ('1', 'Flat'), ('2', 'Down')],
    'thal': [('0', 'Normal'), ('1', 'Fixed'), ('2', 'Reversible')],
    'ca': [('0', '0'), ('1', '1'), ('2', '2'), ('3', '3')]
}

FEATURES_ORDER = ["age","sex","cp","trestbps","chol","fbs","restecg",
                  "thalch","exang","oldpeak","slope","ca","thal"]

# ========== VALIDATION ==========
def validate_input(data):
    errors = []

    if not (0 < data['age'] < 120):
        errors.append("العمر غير منطقي")

    if not (50 < data['trestbps'] < 250):
        errors.append("ضغط الدم غير منطقي")

    if not (100 < data['chol'] < 600):
        errors.append("الكوليسترول غير منطقي")

    if not (0 <= data['oldpeak'] <= 10):
        errors.append("oldpeak غير منطقي")

    return errors

# ========== HTML ==========
html_code = """
<!DOCTYPE html>
<html dir="rtl">
<head>
<title>Heart AI</title>
</head>
<body>

<h2>توقع أمراض القلب</h2>

<form method="POST" action="/predict">
{% for col in cols %}
    <label>{{col}}</label><br>
    {% if col in options %}
        <select name="{{col}}">
        {% for val, txt in options[col] %}
            <option value="{{val}}">{{txt}}</option>
        {% endfor %}
        </select>
    {% else %}
        <input type="number" step="any" name="{{col}}">
    {% endif %}
    <br><br>
{% endfor %}
<button type="submit">Predict</button>
</form>

<hr>

<h3>رفع CSV للتوقع الجماعي</h3>
<form method="POST" action="/batch" enctype="multipart/form-data">
<input type="file" name="file">
<button type="submit">Upload</button>
</form>

{% if result %}
    <h3>{{result}}</h3>
{% endif %}

</body>
</html>
"""

# ========== ROUTES ==========
@app.route('/')
def home():
    return render_template_string(html_code, cols=FEATURES_ORDER, options=OPTIONS, result=None)


# 🔹 Single Prediction
@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = {col: float(request.form.get(col, 0)) for col in FEATURES_ORDER}

        errors = validate_input(data)
        if errors:
            return render_template_string(html_code, cols=FEATURES_ORDER,
                                          options=OPTIONS,
                                          result="❌ " + " | ".join(errors))

        final = np.array([list(data.values())])

        prob = model.predict_proba(final)[0][1]  # probability of disease
        pred = model.predict(final)[0]

        risk_level = "منخفض"
        if prob > 0.7:
            risk_level = "مرتفع"
        elif prob > 0.4:
            risk_level = "متوسط"

        result = f"Prediction: {pred} | Risk: {prob:.2f} | Level: {risk_level}"

        return render_template_string(html_code, cols=FEATURES_ORDER,
                                      options=OPTIONS, result=result)

    except Exception as e:
        return f"خطأ: {e}"


# 🔹 Batch Prediction (CSV)
@app.route('/batch', methods=['POST'])
def batch():
    try:
        file = request.files['file']
        df = pd.read_csv(file)

        if not all(col in df.columns for col in FEATURES_ORDER):
            return "❌ CSV لا يحتوي على الأعمدة المطلوبة"

        preds = model.predict(df[FEATURES_ORDER])
        probs = model.predict_proba(df[FEATURES_ORDER])[:, 1]

        df['Prediction'] = preds
        df['Risk'] = probs

        df.to_csv("results.csv", index=False)

        return "✅ تم التحليل - تم حفظ النتائج في results.csv"

    except Exception as e:
        return f"خطأ: {e}"


# ========== RUN ==========
if __name__ == '__main__':
    from google.colab import output
    output.serve_kernel_port_as_window(5000, path='/')
    app.run(port=5000)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [27/Mar/2026 10:55:22] "GET / HTTP/1.1" 200 -
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
INFO:werkzeug:127.0.0.1 - - [27/Mar/2026 10:57:23] "POST /predict HTTP/1.1" 200 -
